In [39]:
from PIL import Image
import cv2 
import numpy as np 
import matplotlib.pyplot as plt 
from torch.utils.data import DataLoader, Dataset
import os
import h5py
import warnings
import glob
import sys
import tqdm
import random
from joblib import Parallel, delayed

In [11]:
root_path = os.path.abspath("../")
data_path = os.path.join(root_path, "data")
image_path = os.path.join(data_path, "train")

In [12]:
# read in saved image contours
def load_contours_from_hdf5(filepath='/kaggle/input/ecg-image-contours/contours.h5'):
    """
    Load all contours from HDF5 file back into dictionary format
    """
    contour_dict = {}
    
    with h5py.File(filepath, 'r') as f:
        for img_id in f.keys():
            grp = f[img_id]
            
            contour_dict[img_id] = {
                'contour': grp['contour'][:],  # Load the contour array
                'scale_x': grp.attrs['scale_x'],
                'scale_y': grp.attrs['scale_y'], 
                'half': grp.attrs['half']
            }
    
    return contour_dict

# stops us reloading contours every time which is annoyingly slow
if 'contours' not in globals():
    try:
        contours = load_contours_from_hdf5(os.path.join(data_path, "contours.h5"))
    except FileNotFoundError:
        print("Unable to find contours file")

In [13]:
def smart_pad_and_resize_ecg(image, target_size=(224, 224), resize_strategy=cv2.INTER_AREA):
    """
    Resize image to fit within target_size while maintaining aspect ratio,
    then pad with white to reach exact target_size.
    
    Args:
        image: Input image (numpy array or PIL Image) - grayscale or color
        target_size: Tuple of (height, width) for output size
        
    Returns:
        Resized and padded image of exactly target_size dimensions
    """
    if isinstance(image, Image.Image):
        image = np.array(image)
    
    h, w = image.shape[:2]
    target_h, target_w = target_size
    
    # Calculate scaling factor to fit within target size (maintains aspect ratio)
    scale = min(target_h / h, target_w / w)
    
    # Calculate new dimensions
    new_h = int(h * scale)
    new_w = int(w * scale)
    
    # Resize image maintaining aspect ratio
    resized = cv2.resize(image, (new_w, new_h), interpolation=resize_strategy)
    
    # Create white canvas of target size (255 = white)
    if len(image.shape) == 3:
        # Color/3-channel image
        padded = np.full((target_h, target_w, image.shape[2]), 255, dtype=image.dtype)
    else:
        # Grayscale image
        padded = np.full((target_h, target_w), 255, dtype=image.dtype)
    
    # Calculate padding offsets to center the image
    y_offset = (target_h - new_h) // 2
    x_offset = (target_w - new_w) // 2
    
    # Place resized image in center of white canvas
    padded[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    
    return padded

def crop_to_bounding_box(image: np.ndarray, corners: np.ndarray):
    """
    Crop the axis-aligned bounding box defined by corners.
    corners MUST already be in the same coordinate frame as image.
    """

    # get bounding box
    x_min = int(np.floor(corners[:,0].min()))
    y_min = int(np.floor(corners[:,1].min()))
    x_max = int(np.ceil(corners[:,0].max()))
    y_max = int(np.ceil(corners[:,1].max()))

    h, w = image.shape[:2]

    # clip to image bounds
    x_min = max(0, min(x_min, w - 1))
    x_max = max(1, min(x_max, w))
    y_min = max(0, min(y_min, h - 1))
    y_max = max(1, min(y_max, h))

    cropped = image[y_min:y_max, x_min:x_max]
    return cropped

def preproc_pipeline(
        input_image, 
        contour_data,
        BUFFER=10):

    # Convert to grayscale
    gray = cv2.cvtColor(input_image, cv2.COLOR_BGR2GRAY)
    imH, imW = gray.shape

    # Extract and scale contour points
    page = contour_data["contour"].astype(np.int32)
    epsilon = 0.02 * cv2.arcLength(page, True)
    corners = cv2.approxPolyDP(page, epsilon, True)
    corners = np.concatenate(corners).astype(np.float32)

    # Undo half offset and apply scale factors
    corners[:, 0] = (corners[:, 0] - contour_data["half"]) * contour_data["scale_x"]
    corners[:, 1] = (corners[:, 1] - contour_data["half"]) * contour_data["scale_y"]

    # Check if padding is needed
    x_min, y_min = corners.min(axis=0)
    x_max, y_max = corners.max(axis=0)

    pad_left   = max(-x_min, 0) + BUFFER
    pad_top    = max(-y_min, 0) + BUFFER
    pad_right  = max(x_max - imW, 0) + BUFFER
    pad_bottom = max(y_max - imH, 0) + BUFFER

    # If padding needed, pad both image and corners
    if any(v > 0 for v in [pad_left, pad_top, pad_right, pad_bottom]):
        gray = np.pad(gray,
                    ((int(pad_top), int(pad_bottom)),
                    (int(pad_left), int(pad_right))),
                    mode="constant",
                    constant_values=255)
        corners[:, 0] += pad_left
        corners[:, 1] += pad_top

    # Crop to bounding box
    cropped = crop_to_bounding_box(gray, corners)

    return cropped

In [ ]:
# define image dataset that we are going to load

def imread_clean(path):
    """ 
    Image read in that uses PIL Image
    Slower, but doesn't raise eXif warnings that we get with cv2 when using Kaggle
    """
    img = Image.open(path)
    img = img.convert('RGB')  # Strips metadata
    return np.array(img)

def imread_clean(path):
    """
    Read in images using cv2
    Quicker, but raises a very annoying warning on Kaggle that can only be hidden by flushing stdout to a logfile
    """
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def process_single_img(
    img, 
    img_id,
    contour_input=contours,
    desired_aspect_ratio = 0.5, 
    desired_width = 512,
    printing=False
): 
    """
    Processes a single image using our preprocessing pipeline
    """
    image_name = f"train_{str(img_id).zfill(6)}.png"
    # convert image: redundant for cv2, necessary for PIL
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    if printing:
        print("RAW IMAGE STATS")
        test_img_size(img)
        print()
    proc_image = preproc_pipeline(
                        input_image=img, 
                        contour_data=contour_input[image_name.split(".")[0]]
                        )
    if printing:
        print("PROC IMAGE STATS")
        test_img_size(proc_image)
        print()
    resized_image = smart_pad_and_resize_ecg(
        image=proc_image, 
        target_size=(int(desired_width*desired_aspect_ratio), desired_width),
        resize_strategy=cv2.INTER_AREA
    )
    if printing:
        print("RESIZED IMAGE STATS")
        test_img_size(resized_image)
        print()
    return resized_image

def test_img_size(img):
    # Raw RAM usage
    raw_bytes = img.nbytes

    # Compressed size estimates
    _, jpg = cv2.imencode(".jpg", img)
    _, png = cv2.imencode(".png", img)

    print(f"RAM size: {raw_bytes/1024:.2f} KB")
    print(f"JPEG size: {len(jpg)/1024:.2f} KB")
    print(f"PNG size: {len(png)/1024:.2f} KB\n")

In [ ]:
# # Pytorch: I don't actually think we want this!

# class ECGImageDataset(Dataset):
#     def __init__(self, image_paths, verbose_init=True, desired_aspect_ratio=0.5, desired_width=512):
#         super().__init__()
        
#         self.image_paths = image_paths
#         self.path_idx_map = {} 
        
#         for idx, path in enumerate(self.image_paths): 
#             filename = os.path.basename(path)
#             image_id = int(filename.rsplit("_", 1)[-1].split(".")[0])
#             self.path_idx_map[idx] = image_id
            
#         self.desired_aspect_ratio = desired_aspect_ratio
#         self.desired_width = desired_width
            
#         if verbose_init:
#             print(f"Images will be resized to {int(desired_aspect_ratio*desired_width), desired_width}")
        
#     def __len__(self): 
#         return len(self.image_paths)
    
#     def __getitem__(self, idx): 
#         image_path = self.image_paths[idx]
#         image_index = self.path_idx_map[idx]
        
#         # read in the image
#         try: 
#             image = imread_clean(image_path)
#         except Exception as e:
#             print(f"Error in trying to read in image {image_index} at path {image_path}")
            
#         # process the image
#         image_conv = process_single_img(
#                             img=image,
#                             img_id=image_index, 
#                             desired_aspect_ratio=self.desired_aspect_ratio, 
#                             desired_width=self.desired_width, 
#                             printing=False) # avoid printing to avoid I/O noise
        
#         return image_index, image_conv
    
# num_workers = 0 if sys.platform == 'darwin' else 4 # needed because issues with using multiple workers on Mac
# print(f"Using num_workers = {num_workers}")

# dataset = ECGImageDataset(
#                 image_paths=image_list,
#                 verbose_init=True,
#                 desired_aspect_ratio=0.5, 
#                 desired_width=512
#             )

# dataloader = DataLoader( 
#                 dataset,
#                 batch_size=8, 
#                 shuffle=False, 
#                 num_workers=num_workers, 
#                 )

# # iterate over dataloader and propagate list of processed images
# proc_images = []

# print(f"Total number of batches = {len(dataloader)}")
# for enum, (indices, images) in enumerate(dataloader):
#     print(f"Batch number = {enum}")
#     print(indices.shape)
#     print(images.shape)
#     sys.exit()

In [40]:
def single_output(
    image_id,
    image_dirname,
    desired_aspect_ratio=0.5, 
    desired_width=512,
):
    # open image
    img_name = f"train_{str(image_id).zfill(6)}.png"
    img_path = os.path.join(image_dirname, img_name)
    try: 
        image = imread_clean(img_path)
    except Exception as e: 
        print(f"Unable to load image at path {img_path}")
    image_conv = process_single_img(
        img = image, 
        img_id = image_id, 
        desired_aspect_ratio = desired_aspect_ratio, 
        desired_width=desired_width,
        printing=False
    )
    return {img_name: image_conv}

def extract_all_images(
    image_id_list, 
    image_dirname, 
    desired_aspect_ratio, 
    desired_width,
    n_jobs=-1, 
    **joblib_kwargs
):

    print(f"Processing {len(image_id_list)} images")
    print()
    print(f"Expecting to find images at {image_dirname}")
    print()
    
    results = Parallel(n_jobs=n_jobs, **joblib_kwargs)(
        delayed(single_output)(
            image_id, image_dirname, desired_aspect_ratio, desired_width
        ) 
        for image_id in image_id_list 
    )
    
    return results

In [25]:
# define our image list to try to

with open(os.path.join(root_path, "broken_images", "valid_images_list.txt"), "r") as file: 
    image_list = [line.strip() for line in file]

ids = set([int(obj.split(".")[-2][-6:]) for obj in image_list])

# need to filter images so that we only try to use the ones that we have local copies of
# and also change the filenames

local_files = glob.glob(os.path.abspath(os.path.join(data_path, "train", "train_*")))
indices = set([int(item.split(".")[0][-6:]) for item in local_files])

# usable local ids 
local_ids = indices.intersection(ids)

# filter X_train and X_test to only contain the correct ids
image_usable = [elem for elem in image_list if int(elem.split(".")[0][-6:]) in local_ids]
# rewrite paths to actually be the local ones
image_list = [os.path.abspath(os.path.join(data_path, "train", path.split("/")[-1])) for path in image_usable]

print(f"Actual number of local samples: Train = {len(image_list)}")

Actual number of local samples: Train = 4999


In [52]:
results = extract_all_images(image_id_list=list(local_ids)[:10],
                            image_dirname=image_path, 
                            desired_aspect_ratio=0.5,
                            desired_width=512, 
                            verbose=10)

Processing 10 images

Expecting to find images at /Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of  10 | elapsed:    1.2s remaining:    2.9s
[Parallel(n_jobs=-1)]: Done   5 out of  10 | elapsed:    1.6s remaining:    1.6s
[Parallel(n_jobs=-1)]: Done   7 out of  10 | elapsed:    2.1s remaining:    0.9s
[Parallel(n_jobs=-1)]: Done  10 out of  10 | elapsed:    2.9s finished


In [ ]:
def save_as_hdf5(results, output_path):
    images = []
    img_names = []
    
    for result_dict in results:
        for img_name, img_array in result_dict.items():
            images.append(img_array)
            img_names.append(img_name)
    
    images_array = np.stack(images)
    img_names_array = np.array(img_names, dtype="S")
    num_images, H, W = images_array.shape
    
    with h5py.File(output_path, 'w') as f:
        f.create_dataset(
            'images',
            data=images_array,
            compression='gzip',
            chunks=(1, H, W)         
        )
        
        f.create_dataset(
            'img_names',
            data=img_names_array,
            compression='gzip'    
        )

save_as_hdf5(results, "proc_images.h5")